In [2]:
import pandas as pd
import numpy as np
import skfuzzy as fuzz
from typing import Optional, List, Tuple, Dict, Any


def run_fcm_on_excel_numeric_columns(
    fcm_inputs_path: str = "FCM_INPUTS.xlsx",
    #fcm_inputs_path: str = "weekly_trends_sg.xlsx",
    #fcm_inputs_path: str = "Day.xlsx",
    sheet_name: str = "WEEKLY_FEATURES",
    #heet_name: str = "Sheet1",
    c: int = 3,
    m: float = 2.0,
    error: float = 1e-6,
    maxiter: int = 1000,
    seed: int = 42,
    min_samples: int = 30,
    exclude_cols: Optional[List[str]] = None,
) -> List[Tuple[str, np.ndarray, Dict[str, Any]]]:
    df = pd.read_excel(fcm_inputs_path, sheet_name=sheet_name)

    if exclude_cols:
        df = df.drop(
            columns=[col for col in exclude_cols if col in df.columns],
            errors="ignore"
        )

    numeric_columns = df.select_dtypes(include=[np.number]).columns

    results: List[Tuple[str, np.ndarray, Dict[str, Any]]] = []

    for col in numeric_columns:
        values = df[col].dropna().values.astype(np.float64)
        n = int(len(values))

        if n < min_samples:
            continue

        data = values.reshape(1, -1)

        cntr, u, u0, d, jm, p, fpc = fuzz.cluster.cmeans(
            data=data,
            c=c,
            m=m,
            error=error,
            maxiter=maxiter,
            init=None,
            seed=seed
        )

        centers_sorted = np.sort(cntr.flatten())

        # --- jm/fpc summary ---
        jm_arr = np.asarray(jm, dtype=np.float64)
        n_iter = int(jm_arr.size)

        jm_first = float(jm_arr[0]) if n_iter > 0 else np.nan
        jm_last = float(jm_arr[-1]) if n_iter > 0 else np.nan

        jm_delta = float(jm_first - jm_last) if np.isfinite(jm_first) and np.isfinite(jm_last) else np.nan
        jm_delta_pct = (
            float((jm_delta / jm_first) * 100.0)
            if np.isfinite(jm_delta) and np.isfinite(jm_first) and jm_first != 0
            else np.nan
        )

        summary: Dict[str, Any] = {
            "n_samples": n,
            "c": c,
            "m": m,
            "n_iter": n_iter,
            "jm_first": jm_first,
            "jm_last": jm_last,
            "jm_delta": jm_delta,
            "jm_delta_pct": jm_delta_pct,
            "fpc": float(fpc),
        }

        # --- Konsol çıktısı ---
        print(f"Sütun: {col}")
        print(f"  N (örnek):          {n}")
        print(f"  Iterasyon sayısı:   {n_iter}")
        print(f"  jm (ilk):           {jm_first:.6f}" if np.isfinite(jm_first) else "  jm (ilk):           NaN")
        print(f"  jm (son):           {jm_last:.6f}" if np.isfinite(jm_last) else "  jm (son):           NaN")
        if np.isfinite(jm_delta_pct):
            print(f"  jm düşüşü (%):      {jm_delta_pct:.2f}")
        else:
            print("  jm düşüşü (%):      NaN")
        print(f"  fpc:                {float(fpc):.6f}")

        for i, center in enumerate(centers_sorted, start=1):
            print(f"  Küme {i}: {center:.12f}")
        print("-" * 40)

        results.append((col, centers_sorted, summary))

    return results


# ============================================================
# MAIN FLOW – runs only FCM
# ============================================================

def main():
    run_fcm_on_excel_numeric_columns(
        fcm_inputs_path="FCM_INPUTS.xlsx",
        #fcm_inputs_path="weekly_trends_sg.xlsx",
        #fcm_inputs_path="Day.xlsx",
        sheet_name="WEEKLY_FEATURES",
        #sheet_name="Sheet1",
        c=3,
        m=2.0,
        error=1e-6,
        maxiter=1000,
        seed=42,
        min_samples=30,
    )


if __name__ == "__main__":
    main()


Sütun: n_days
  N (örnek):          225
  Iterasyon sayısı:   3
  jm (ilk):           0.000000
  jm (son):           0.000000
  jm düşüşü (%):      -893.22
  fpc:                0.333333
  Küme 1: 5.000000000000
  Küme 2: 5.000000000000
  Küme 3: 5.000000000000
----------------------------------------
Sütun: week_avg
  N (örnek):          225
  Iterasyon sayısı:   55
  jm (ilk):           33029.425127
  jm (son):           9340.681904
  jm düşüşü (%):      71.72
  fpc:                0.785517
  Küme 1: 41.705038863946
  Küme 2: 68.792979582001
  Küme 3: 89.847526311643
----------------------------------------
Sütun: plant_week_avg
  N (örnek):          225
  Iterasyon sayısı:   53
  jm (ilk):           775.795150
  jm (son):           257.319255
  jm düşüşü (%):      66.83
  fpc:                0.855607
  Küme 1: 61.036681443257
  Küme 2: 66.787949802650
  Küme 3: 71.560600367698
----------------------------------------
Sütun: ope_week_avg
  N (örnek):          225
  Iterasyon sayısı: 